## EXP 1: Image processing (OpenCV)

In [ ]:
"""
Aim: To perform basic image processing operations using Python and OpenCV and analyze the effects of histogram equalization,
edge detection, thresholding, image augmentation, and morphological operations.

Algorithm:

1. Import the required libraries and load the input image.
2. Apply histogram equalization to enhance the image contrast.
3. Compute and display the histograms of the original and equalized images.
4. Apply Gaussian Blur and perform Canny edge detection.
5. Perform image augmentation using rotation, zoom and horizontal flipping.
6. Convert the image into a binary image using thresholding.
7. Apply erosion and dilation using a structuring kernel.
8. Display the original and all processed images for comparison.

Result:

Thus, basic image processing operations were successfully implemented using Python and OpenCV, producing the expected enhanced
and transformed images.
"""

import cv2, numpy as np, matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator

path = r"Image_1.jfif"
img = cv2.imread(path, 0)

def show(imgs, titles, rows, cols, size=(10, 8)):
    plt.figure(figsize=size)
    for i, (im, t) in enumerate(zip(imgs, titles), 1):
        plt.subplot(rows, cols, i); plt.imshow(im, cmap="gray"); plt.title(t); plt.axis("off")
    plt.tight_layout(); plt.show()

equ = cv2.equalizeHist(img)
show([img, equ], ["Original", "Equalized"], 1, 2)
for i, (im, t) in enumerate([(img, "Original"), (equ, "Equalized")], 1):
    plt.subplot(1, 2, i); plt.hist(im.ravel(), 256, [0, 256]); plt.title(t + " Histogram")
plt.show()

edges = cv2.Canny(cv2.GaussianBlur(img, (5, 5), 0), 100, 200)
show([img, edges], ["Original", "Edges"], 1, 2)

_, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
k = np.ones((5, 5), np.uint8)
show([img, binary, cv2.erode(binary, k), cv2.dilate(binary, k)],
     ["Original", "Binary Threshold", "Erosion", "Dilation"], 2, 2)

rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)[None]
gen = ImageDataGenerator(rotation_range=40, zoom_range=0.2, horizontal_flip=True).flow(rgb, batch_size=1)
show([rgb[0]] + [next(gen)[0].astype("uint8") for _ in range(8)],
     ["Original"] + [f"Augmented {i}" for i in range(1, 9)], 3, 3, (10, 10))

## EXP 2: ANN on MNIST

In [ ]:
"""
Aim: To build and train an Artificial Neural Network (ANN) using TensorFlow and Keras for handwritten digit classification on
the MNIST dataset, and to evaluate it using accuracy, loss and training-validation curves.

Algorithm:

1. Import the required TensorFlow, Keras and Matplotlib libraries.
2. Load the MNIST dataset and normalize the pixel values to [0, 1].
3. Build a Sequential ANN with a Flatten layer, one hidden Dense layer (64, ReLU) and an output Dense layer (10, softmax).
4. Compile with the Adam optimizer and sparse categorical cross-entropy loss.
5. Train for 10 epochs with batch size 32 and 20% validation data.
6. Evaluate the model on the test data to get the loss and accuracy.
7. Plot the training and validation loss curves.
8. Plot the training and validation accuracy curves and analyze the performance.

Result:

Thus, the ANN model was successfully trained on the MNIST dataset and achieved high accuracy in handwritten digit classification.
The loss and accuracy curves showed effective learning and good model performance.
"""

import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
tf.keras.utils.set_random_seed(42)

(xtr, ytr), (xte, yte) = tf.keras.datasets.mnist.load_data()
xtr, xte = xtr / 255.0, xte / 255.0

model = Sequential([
    Input((28, 28)), Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
h = model.fit(xtr, ytr, epochs=10, batch_size=32, validation_split=0.2)
loss, acc = model.evaluate(xte, yte, verbose=0)
print("Test Loss:", loss, "| Test Accuracy:", acc)

plt.figure(figsize=(12, 5))
for i, k in enumerate(["loss", "accuracy"], 1):
    plt.subplot(1, 2, i); plt.plot(h.history[k], label="Training"); plt.plot(h.history["val_" + k], label="Validation")
    plt.title(k.title() + " Curve"); plt.xlabel("Epoch"); plt.legend()
plt.show()

## EXP 3: CNN on EMNIST Balanced

In [ ]:
"""
Aim: To build and train a CNN using TensorFlow and Keras for handwritten character classification on the EMNIST Balanced dataset,
and to evaluate it using accuracy, loss curves and a sample prediction.

Algorithm:

1. Import NumPy, Matplotlib, TensorFlow and gzip, then read the EMNIST Balanced files.
2. Normalize the pixel values and reshape the images to 28 x 28 x 1.
3. Build a Sequential CNN with two Conv2D + MaxPooling blocks, Flatten, Dense (128) and Dropout (0.5), and a Dense output layer (47).
4. Compile with the Adam optimizer and sparse categorical cross-entropy loss.
5. Train for 4 epochs with batch size 64 and 20% validation data.
6. Evaluate the model on the test data to get the loss and accuracy.
7. Plot the training and validation accuracy and loss curves.
8. Predict one test image, convert the label to a character using the mapping file, and show the result with its confidence.

Result:

Thus, the CNN model was successfully trained on the EMNIST Balanced dataset and achieved good accuracy in classifying handwritten
characters across 47 classes. The curves confirmed effective learning.
"""

import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
tf.keras.utils.set_random_seed(42)
import gzip

def idx(split, kind, off):
    return np.frombuffer(gzip.open(f"emnist-balanced-{split}-{kind}-ubyte.gz").read(), np.uint8, offset=off)

xtr, ytr = idx("train", "images-idx3", 16).reshape(-1, 28, 28, 1) / 255.0, idx("train", "labels-idx1", 8)
xte, yte = idx("test", "images-idx3", 16).reshape(-1, 28, 28, 1) / 255.0, idx("test", "labels-idx1", 8)
print("Train:", xtr.shape, "| Test:", xte.shape, "| Classes:", len(np.unique(ytr)))

model = Sequential([
    Input((28, 28, 1)),
    Conv2D(32, 3, activation="relu"), MaxPooling2D(),
    Conv2D(64, 3, activation="relu"), MaxPooling2D(),
    Flatten(), Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(47, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
h = model.fit(xtr, ytr, epochs=4, batch_size=64, validation_split=0.2)
loss, acc = model.evaluate(xte, yte, verbose=0)
print("Test Loss:", loss, "| Test Accuracy:", acc)

plt.figure(figsize=(12, 5))
for i, k in enumerate(["loss", "accuracy"], 1):
    plt.subplot(1, 2, i); plt.plot(h.history[k], label="Training"); plt.plot(h.history["val_" + k], label="Validation")
    plt.title(k.title() + " Curve"); plt.xlabel("Epoch"); plt.legend()
plt.show()
mp = {int(a): chr(int(b)) for a, b in (l.split() for l in open("emnist-balanced-mapping.txt"))}
i = 25
p = model.predict(xte[i:i + 1], verbose=0)[0]
plt.imshow(xte[i].squeeze(), cmap="gray"); plt.axis("off"); plt.show()
print("Actual    :", mp[yte[i]], "\nPredicted :", mp[p.argmax()], "\nConfidence:", round(p.max() * 100, 2), "%")

## EXP 4: CNN face recognition (LFW)

In [ ]:
"""
Aim: To develop a Convolutional Neural Network (CNN) model using TensorFlow to recognize and classify different persons from
facial images in the LFW (Labeled Faces in the Wild) dataset.

Algorithm:

1. Import NumPy, TensorFlow, Matplotlib and Scikit-learn libraries.
2. Load the LFW face dataset with a minimum of 70 images per person.
3. Extract the face images and the target labels.
4. Normalize the pixel values and add the channel dimension.
5. Split the data into 80% training and 20% testing using stratified sampling.
6. Build a CNN using Conv2D, ReLU, MaxPooling, Flatten and Dense layers.
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss, then train for 10 epochs.
8. Evaluate on the test data and predict the identity of a sample test face.

Result:

Thus, a CNN-based face recognition model was successfully implemented using the LFW dataset. The trained model was evaluated on
the test data and predicted the identity of a sample facial image.
"""

import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
tf.keras.utils.set_random_seed(42)
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split

lfw = fetch_lfw_people(min_faces_per_person=70, resize=0.5)
X, y, names = lfw.images[..., None] / 255.0, lfw.target, lfw.target_names
for n, c in zip(names, np.bincount(y)): print(n, ":", c, "images")
print("Total images:", len(X), "| Total people:", len(names))

xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = Sequential([
    Input(X.shape[1:]),
    Conv2D(32, 3, activation="relu"), MaxPooling2D(),
    Conv2D(64, 3, activation="relu"), MaxPooling2D(),
    Flatten(), Dense(128, activation="relu"),
    Dense(len(names), activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
h = model.fit(xtr, ytr, epochs=10, batch_size=32, validation_split=0.2)
loss, acc = model.evaluate(xte, yte, verbose=0)
print("Test Loss:", loss, "| Test Accuracy:", acc)

p = model.predict(xte[:1], verbose=0).argmax()
print("Predicted person:", names[p], "\nActual person   :", names[yte[0]])
plt.imshow(xte[0].squeeze(), cmap="gray"); plt.title("Predicted: " + names[p]); plt.axis("off"); plt.show()

## EXP 5: Char-level text generation (SimpleRNN)

In [ ]:
"""
Aim: To build a character-level language model using a SimpleRNN in TensorFlow. The model learns from the Shakespeare text 
and generates new text from a starting phrase.

Algorithm:

1. Import TensorFlow and NumPy, then load the Shakespeare text file.
2. Find all unique characters and give each one a number (vocabulary).
3. Convert the whole text into numbers using this mapping.
4. Cut the text into sequences of 100 characters. The target is the same sequence shifted by one character.
5. Shuffle the sequences and group them into batches of 64.
6. Build the model with an Embedding layer (128), a SimpleRNN layer (256 units) and a Dense output layer (65 units).
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss (with logits), then train for 10 epochs.
8. Start with "Shall ", predict one character at a time using the last 100 characters, and add each prediction to the text.

Result:

Thus, a character-level language model using SimpleRNN was successfully developed and trained on the given text corpus. The trained model
successfully generated new text based on the learned character patterns.
"""

import numpy as np, tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy
tf.keras.utils.set_random_seed(42)

text = open(r"shakespeare.txt", encoding="utf-8").read()
vocab = sorted(set(text)); V = len(vocab)
c2i = {c: i for i, c in enumerate(vocab)}
ids = np.array([c2i[c] for c in text], dtype="int32")
print("Corpus length:", len(text), "| Vocab size:", V)

w = np.lib.stride_tricks.sliding_window_view(ids, 101)
ds = tf.data.Dataset.from_tensor_slices((w[:, :-1], w[:, 1:])).shuffle(10000).batch(64, drop_remainder=True)

model = Sequential([
    Embedding(V, 128),
    SimpleRNN(256, return_sequences=True),
    Dense(V)])
model.compile(optimizer="adam", loss=SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
model.summary()
model.fit(ds, epochs=10)

def generate(start, n=200):
    out = start
    for _ in range(n):
        x = np.array([[c2i[c] for c in out[-100:]]])
        out += vocab[model.predict(x, verbose=0)[0, -1].argmax()]
    return out

print("\nGenerated Text:\n", generate("Shall ", 200))

## EXP 6: IMDB sentiment (LSTM)

In [ ]:
"""
Aim: To build an LSTM model that classifies IMDb movie reviews as Positive or Negative, and to test it on a custom sentence.

Algorithm:

1. Import TensorFlow and load the IMDb data, keeping only the top 10,000 most common words.
2. Pad every review to a fixed length of 200 words.
3. Build the model with an Embedding layer (10,000 × 128), an LSTM layer (64 units) and a Dense layer (1 unit, sigmoid).
4. Compile with the Adam optimizer, binary cross-entropy loss and accuracy as the metric.
5. Train for 3 epochs with batch size 64.
6. Evaluate on the test data and print the test accuracy.
7. Load the IMDb word index and convert the sentence "This movie was super" into numbers, then pad it to 200.
8. Predict the score. If the score is 0.5 or more, the sentiment is Positive, otherwise Negative.

Result:

Thus, an LSTM-based sentiment analysis model was successfully developed and trained using the IMDb movie review data. The trained model 
successfully classified movie reviews into positive or negative sentiments.
"""

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
tf.keras.utils.set_random_seed(42)

(xtr, ytr), (xte, yte) = imdb.load_data(num_words=10000)
xtr, xte = pad_sequences(xtr, maxlen=200), pad_sequences(xte, maxlen=200)

model = Sequential([
    Embedding(10000, 128),
    LSTM(64),
    Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(xtr, ytr, epochs=3, batch_size=64)
print("Test Accuracy:", model.evaluate(xte, yte)[1])

wi = imdb.get_word_index()
def encode(sentence):
    ids = [1]
    for w in sentence.lower().split():
        i = wi.get(w.strip(".,!?"), None)
        ids.append(i + 3 if i is not None and i + 3 < 10000 else 2)
    return pad_sequences([ids], maxlen=200)

sentence = "This movie was super"
p = model.predict(encode(sentence))[0][0]
print("Sentence:", sentence, "\nSentiment:", "Positive" if p >= 0.5 else "Negative", "\nScore:", p)

## EXP 7: POS tagging (Encoder-Decoder LSTM)

In [ ]:
"""
Aim: To build an Encoder–Decoder LSTM model that gives each word in a sentence its POS tag, using the Penn Treebank corpus.

Algorithm:

1. Load the tagged sentences from the Penn Treebank and split each into a list of words and a list of tags.
2. Split the data into 80% training and 20% testing.
3. Create a word tokenizer (using the training data only) and a tag tokenizer, and convert both to numbers.
4. Pad all sequences to the longest training sentence length.
5. Build the Encoder with an Embedding layer (128) and an LSTM layer (128 units) that returns its outputs and states.
6. Build the Decoder with an LSTM layer (128 units) that starts from the encoder states, followed by a Dense softmax layer that predicts
   a tag for each word.
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss, then train for 5 epochs with batch size 32 and 10% validation data.
8. Evaluate on the test data, take the highest-probability tag for each word, and compare predicted and actual tags.

Result:

Thus, an Encoder–Decoder LSTM model for Part-of-Speech tagging was successfully developed and trained using the Penn Treebank corpus. 
The trained model successfully predicted the appropriate Part-of-Speech tags for the given sentences.
"""

import nltk, numpy as np
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
nltk.download("treebank")

sents = treebank.tagged_sents()
X = [[w for w, t in s] for s in sents]
Y = [[t for w, t in s] for s in sents]
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.2, random_state=42)

wtok = Tokenizer(lower=True, oov_token="<UNK>"); wtok.fit_on_texts(Xtr)     
ttok = Tokenizer(lower=False, filters="");       ttok.fit_on_texts(Ytr)
wv, tv = len(wtok.word_index) + 1, len(ttok.word_index) + 1
max_len = max(len(s) for s in Xtr)

def pad(tok, data):
    return pad_sequences(tok.texts_to_sequences(data), maxlen=max_len, padding="post", truncating="post")
Xtr_p, Xte_p, Ytr_p, Yte_p = pad(wtok, Xtr), pad(wtok, Xte), pad(ttok, Ytr), pad(ttok, Yte)
print("Vocab:", wv, "| Tags:", tv, "| Max len:", max_len, "| Input:", Xtr_p.shape)

inp = Input((max_len,))
enc_out, h, c = LSTM(128, return_sequences=True, return_state=True)(Embedding(wv, 128, mask_zero=True)(inp))
dec_out = LSTM(128, return_sequences=True)(enc_out, initial_state=[h, c])
model = Model(inp, Dense(tv, activation="softmax")(dec_out))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit(Xtr_p, Ytr_p[..., None], validation_split=0.1, epochs=5, batch_size=32)
loss, acc = model.evaluate(Xte_p, Yte_p[..., None])
print("Test Loss:", loss, "| Test Accuracy:", acc)

id2tag = {v: k for k, v in ttok.word_index.items()}
pred = model.predict(Xte_p).argmax(-1)
i = 782                                                         
print("\nWords:", Xte[i], "\nActual:", Yte[i], "\nPredicted:", [id2tag.get(t, "UNK") for t in pred[i][:len(Xte[i])]])

## EXP 8: English -> Tamil translation (Seq2Seq LSTM)

In [ ]:
"""
Aim: To build an Encoder–Decoder LSTM model that translates English sentences into Tamil after training on paired sentences.

Algorithm:

1. Read the English–Tamil sentence pairs from the tab-separated file, and remove empty lines and duplicate pairs.
2. Add <start> and <end> tokens to every Tamil sentence.
3. Create separate tokenizers for English and Tamil, convert the sentences to numbers, and pad them to the maximum length.
4. Make the decoder input from the Tamil sequence without its last token, and the decoder output from the sequence 
   without its first token (shifted by one).
5. Build the Encoder with an Embedding layer (64) and an LSTM layer (128 units), and pass its hidden and cell states to the Decoder.
6. Build the Decoder with an Embedding layer (64), an LSTM layer (128 units) and a Dense softmax layer.
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss, then train for 100 epochs with batch size 32.
8. To translate, start with <start>, predict one Tamil word at a time, and stop when <end> is generated.

Result:

Thus, an LSTM-based English-to-Tamil neural machine translation model was successfully developed and trained using paired sentence 
data. The trained model successfully generated Tamil translations for the given English test sentences.
"""

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

pairs = []
for line in open(r"eng_tamil_pairs.txt", encoding="utf-8"):
    p = line.strip().split("\t", 1)
    if len(p) == 2: pairs.append((p[0].strip(), p[1].strip()))
print("Pairs:", len(pairs))
pairs = list(dict.fromkeys(pairs))                              
print("Unique pairs:", len(pairs))
english = [e for e, t in pairs]
tamil = [f"<start> {t} <end>" for e, t in pairs]

def tokenize(texts):
    tok = Tokenizer(filters=""); tok.fit_on_texts(texts)
    seq = tok.texts_to_sequences(texts); L = max(len(s) for s in seq)
    return tok, pad_sequences(seq, maxlen=L, padding="post"), L
et, eng, max_eng = tokenize(english)
tt, tam, max_tam = tokenize(tamil)
ev, tv = len(et.word_index) + 1, len(tt.word_index) + 1
print("Max len:", max_eng, max_tam, "| Vocab:", ev, tv)

dec_in_data, dec_out_data = tam[:, :-1], tam[:, 1:]             

enc_in = Input((max_eng,))
_, h, c = LSTM(128, return_state=True)(Embedding(ev, 64, mask_zero=True)(enc_in))   
dec_in = Input((max_tam - 1,))
dec = LSTM(128, return_sequences=True)(Embedding(tv, 64, mask_zero=True)(dec_in), initial_state=[h, c])  
model = Model([enc_in, dec_in], Dense(tv, activation="softmax")(dec))
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
model.fit([eng, dec_in_data], dec_out_data[..., None], epochs=100, batch_size=32, verbose=1)

rev = {i: w for w, i in tt.word_index.items()}
start, end = tt.word_index["<start>"], tt.word_index["<end>"]
def translate(sentence):
    src = pad_sequences(et.texts_to_sequences([sentence]), maxlen=max_eng, padding="post")
    tgt = [start]
    for _ in range(max_tam - 1):                                
        t = pad_sequences([tgt], maxlen=max_tam - 1, padding="post")
        nxt = model.predict([src, t], verbose=0)[0, len(tgt) - 1].argmax()
        if nxt in (0, end): break
        tgt.append(nxt)
    return " ".join(rev[i] for i in tgt[1:])

for s in ["Get down.", "Goodbye and take care.", "Let's meet again soon.", "Good morning.", "Thank you.", "How are you?", "hospital"]:
    print("\nEnglish :", s, "\nTamil   :", translate(s))

## EXP 9: GAN data augmentation (Fashion-MNIST)

In [ ]:
"""
Aim: To build a GAN that generates synthetic Fashion-MNIST images, and to check whether adding these images to the training data improves classifier accuracy.

Algorithm:

1. Load the Fashion-MNIST dataset. Scale the images to [-1, 1] for the GAN and to [0, 1] for the classifier.
2. Build the Generator, which turns a 100-value noise vector into a 28×28 image.
3. Build the Discriminator, which uses Conv2D, LeakyReLU, Dropout and a sigmoid output to decide if an image is real or fake.
4. Join the Generator and Discriminator into a GAN model, keeping the Discriminator frozen inside it.
5. Train for 50 epochs with batch size 128. In each step, train the Discriminator on real images (label 0.9) and fake images (label 0), 
   then train the Generator to make the Discriminator say "real".
6. Display 16 generated images, then generate 5,000 synthetic images.
7. Train a CNN classifier on the original data only (3 epochs), then train an identical CNN on the original data plus the 5,000 synthetic 
   images (labelled using the first classifier).
8. Test both classifiers on the test set, then compare their accuracies and show them in a bar chart.

Result:

Thus, a GAN-based synthetic data generation and data augmentation model was successfully developed and trained using the Fashion-MNIST 
dataset. The trained GAN successfully generated synthetic images, which were used to augment the training data for image classification.
"""

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Dense, BatchNormalization, LeakyReLU, Reshape,
    Conv2DTranspose, Conv2D, Dropout, Flatten, MaxPooling2D
)

tf.keras.utils.set_random_seed(42)

(xr, ytr), (xt, yte) = tf.keras.datasets.fashion_mnist.load_data()
xr, xt = xr[..., None].astype("float32"), xt[..., None].astype("float32")
x_gan = (xr - 127.5) / 127.5                                    
x_cls, x_test = xr / 255.0, xt / 255.0                          

G = Sequential([Input((100,)),
    Dense(7 * 7 * 256, use_bias=False), BatchNormalization(), LeakyReLU(), Reshape((7, 7, 256)),
    Conv2DTranspose(128, 4, 2, padding="same", use_bias=False), BatchNormalization(), LeakyReLU(),
    Conv2DTranspose(64, 4, 2, padding="same", use_bias=False), BatchNormalization(), LeakyReLU(),
    Conv2D(1, 3, padding="same", activation="tanh")])

D = Sequential([Input((28, 28, 1)),            
    Conv2D(64, 4, 2, padding="same"), LeakyReLU(0.2), Dropout(0.3),
    Conv2D(128, 4, 2, padding="same"), LeakyReLU(0.2), Dropout(0.3),
    Flatten(), Dense(1, activation="sigmoid")])
D.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

D.trainable = False                                             
GAN = Sequential([G, D])
GAN.compile(optimizer=tf.keras.optimizers.Adam(0.0002, 0.5), loss="binary_crossentropy")

noise = lambda n: np.random.normal(0, 1, (n, 100)).astype("float32")
for epoch in range(50):
    for i in range(0, len(x_gan), 128):
        real = x_gan[i:i + 128]; n = len(real)
        fake = G(noise(n), training=True).numpy()               
        D.trainable = True
        D.train_on_batch(real, np.full((n, 1), 0.9))           
        D.train_on_batch(fake, np.zeros((n, 1)))
        D.trainable = False
        GAN.train_on_batch(noise(n), np.ones((n, 1)))           
    print("Epoch:", epoch + 1)

gen = (G.predict(noise(16), verbose=0) + 1) / 2                 
plt.figure(figsize=(8, 8))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(gen[i].squeeze(), cmap="gray"); plt.axis("off")
plt.suptitle("GAN Generated Images"); plt.tight_layout(); plt.show()

num_fake = 5000
fake_images = (G.predict(noise(num_fake), verbose=0) + 1) / 2
print("Generated images:", fake_images.shape)

def classifier():
    m = Sequential([Input((28, 28, 1)),
        Conv2D(32, 3, activation="relu"), MaxPooling2D(),
        Conv2D(64, 3, activation="relu"), MaxPooling2D(),
        Flatten(), Dense(64, activation="relu"), Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

print("Training original classifier...")
m_orig = classifier(); m_orig.fit(x_cls, ytr, epochs=3, batch_size=128)
acc_orig = m_orig.evaluate(x_test, yte, verbose=0)[1]

fake_labels = m_orig.predict(fake_images, verbose=0).argmax(1)

x_aug = np.concatenate([x_cls, fake_images]); y_aug = np.concatenate([ytr, fake_labels])
idx = np.random.permutation(len(x_aug)); x_aug, y_aug = x_aug[idx], y_aug[idx]
print("Augmented data:", x_aug.shape)

print("Training augmented classifier...")
m_aug = classifier(); m_aug.fit(x_aug, y_aug, epochs=3, batch_size=128)
acc_aug = m_aug.evaluate(x_test, yte, verbose=0)[1]

print(f"\nOriginal Accuracy: {acc_orig*100:.2f} %\nGAN Augmented Accuracy: {acc_aug*100:.2f} %\nChange: {(acc_aug-acc_orig)*100:.2f} %")
plt.bar(["Original", "GAN Augmented"], [acc_orig * 100, acc_aug * 100])
plt.ylabel("Accuracy (%)"); plt.title("Classification Accuracy"); plt.ylim(0, 100); plt.show()

## EXP 10: Recommendation system (embeddings)

In [ ]:
"""
Aim: To build a neural network recommendation system using embeddings that suggests products to customers from their past
purchases in sales data.

Algorithm:

1. Load the sales data, remove rows with missing values, and keep only completed orders with quantity greater than 0.
2. Make unique customer-product pairs (purchases) and give each customer and product a number. These pairs get label 1.
3. For every customer, pick up to 4 random products they did not buy as negative samples with label 0.
4. Split the data into 80% training and 20% validation using stratified sampling.
5. Build the model: Embedding (32) for the customer and for the product, join them, then Dense (64), Dropout (0.2), Dense (32) and
   a Dense sigmoid output.
6. Compile with the Adam optimizer, binary cross-entropy loss and AUC as the metric, then train for 30 epochs with batch size 64.
7. For a customer, score all products they have not bought and sort the scores from highest to lowest.
8. Display the top 5 products with their names, categories and recommendation scores.

Result:

Thus, an embedding-based product recommendation model was successfully developed and trained using the sales data. The trained
model successfully recommended the top products for the given customer.
"""

import numpy as np, tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, Embedding, Concatenate
tf.keras.utils.set_random_seed(42)
import pandas as pd
from sklearn.model_selection import train_test_split

cols = ["customer_id", "product_id", "product_name", "product_category", "quantity", "order_status"]
df = pd.read_csv("sales.csv").dropna(subset=cols)
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df = df[(df["order_status"].astype(str).str.strip().str.lower() == "completed") & (df["quantity"] > 0)]

pos = df[["customer_id", "product_id"]].drop_duplicates()
pos["u"], users = pd.factorize(pos["customer_id"])
pos["i"], items = pd.factorize(pos["product_id"])
nu, ni = len(users), len(items)

rng = np.random.default_rng(42)
bought = pos.groupby("u")["i"].apply(set)
neg = [(u, i) for u, s in bought.items()
       for i in rng.choice(list(set(range(ni)) - s), min(4, ni - len(s)), replace=False)]
data = pd.concat([pos[["u", "i"]].assign(y=1.0), pd.DataFrame(neg, columns=["u", "i"]).assign(y=0.0)])
tr, va = train_test_split(data, test_size=0.2, random_state=42, stratify=data["y"])
xy = lambda d: ([d["u"].values, d["i"].values], d["y"].values)

u_in, i_in = Input((1,)), Input((1,))
emb = lambda n, x: Flatten()(Embedding(n, 32)(x))
x = Dense(64, activation="relu")(Concatenate()([emb(nu, u_in), emb(ni, i_in)]))
x = Dense(32, activation="relu")(Dropout(0.2)(x))
model = Model([u_in, i_in], Dense(1, activation="sigmoid")(x))
model.compile("adam", "binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
model.fit(*xy(tr), validation_data=xy(va), epochs=30, batch_size=64)

prod = df.drop_duplicates("product_id").set_index("product_id")
def recommend(cust, n=5):
    u = users.get_loc(cust)
    cand = np.array(sorted(set(range(ni)) - bought.get(u, set())))
    if len(cand) == 0: return pd.DataFrame()
    s = model.predict([np.full(len(cand), u), cand], verbose=0).ravel()
    top = np.argsort(s)[::-1][:n]
    out = prod.loc[items[cand[top]], ["product_name", "product_category"]].reset_index()
    out["score"] = s[top].round(4)
    return out

c = users[10]
print("Customer:", c); print(recommend(c).to_string(index=False))

## EXP 11: Image captioning (InceptionV3 + LSTM, Flickr8k)

In [ ]:
"""
Aim: To build an image captioning model that generates a sentence for an image, using InceptionV3 image features and an LSTM
decoder trained on the Flickr8k dataset.

Algorithm:

1. Load the train, validation and test image names and the captions of the Flickr8k dataset.
2. Clean each caption (lowercase, remove punctuation) and add the words startseq and endseq.
3. Extract a 2048-value feature vector for every image using the pre-trained InceptionV3 model.
4. Build a tokenizer from the training captions and find the vocabulary size and the maximum caption length.
5. Create training data: for each caption, the image features and the first i words are the input and the next word is the output.
6. Build the model: image features go through Dropout and Dense (256), the caption goes through Embedding, Dropout and LSTM (256).
   Add both, then Dense (256) and a Dense softmax output.
7. Compile with the Adam optimizer and sparse categorical cross-entropy loss, then train for 5 epochs with batch size 256.
8. To caption an image, start with startseq, predict one word at a time, and stop at endseq or the maximum length.

Result:

Thus, an image captioning model was successfully developed and trained using the Flickr8k dataset. The trained model generated
a caption for the given test image.
"""

import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Embedding, LSTM, add
tf.keras.utils.set_random_seed(42)
import os, string
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

img_dir, txt_dir = r"Flicker8k_Dataset", r"Flickr8k_text"
names = lambda f: [n for n in (l.strip() for l in open(f"{txt_dir}/{f}", encoding="utf-8")) if os.path.isfile(f"{img_dir}/{n}")]
train_n, val_n, test_n = names("Flickr_8k.trainImages.txt"), names("Flickr_8k.devImages.txt"), names("Flickr_8k.testImages.txt")
print("Train:", len(train_n), "| Val:", len(val_n), "| Test:", len(test_n))

caps = {}
for l in open(f"{txt_dir}/Flickr8k.token.txt", encoding="utf-8"):
    p = l.strip().split("\t")
    if len(p) == 2:
        c = p[1].lower().translate(str.maketrans("", "", string.punctuation))
        caps.setdefault(p[0].split("#")[0], []).append(f"startseq {c} endseq")

inception = InceptionV3(weights="imagenet", pooling="avg")
feat = lambda n: inception.predict(preprocess_input(img_to_array(load_img(f"{img_dir}/{n}", target_size=(299, 299)))[None]), verbose=0)[0]
feats = {n: feat(n) for n in train_n + val_n + test_n}

train_caps = [c for n in train_n for c in caps[n]]
tok = Tokenizer(); tok.fit_on_texts(train_caps)
V, L = len(tok.word_index) + 1, max(len(c.split()) for c in train_caps)
print("Vocabulary size:", V, "| Max caption length:", L)

def seqs(ns):
    X1, X2, y = [], [], []
    for n in ns:
        for c in caps[n]:
            s = tok.texts_to_sequences([c])[0]
            for i in range(1, len(s)):
                X1.append(feats[n]); X2.append(s[:i]); y.append(s[i])
    return np.array(X1), pad_sequences(X2, maxlen=L), np.array(y)
X1, X2, y = seqs(train_n)
X1v, X2v, yv = seqs(val_n)

i1, i2 = Input((2048,)), Input((L,))
a = Dense(256, activation="relu")(Dropout(0.3)(i1))
b = LSTM(256)(Dropout(0.3)(Embedding(V, 256, mask_zero=True)(i2)))
model = Model([i1, i2], Dense(V, activation="softmax")(Dense(256, activation="relu")(add([a, b]))))
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
model.summary()
h = model.fit([X1, X2], y, validation_data=([X1v, X2v], yv), epochs=5, batch_size=256)
plt.plot(h.history["loss"], label="Training Loss"); plt.plot(h.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Image Captioning Model Loss"); plt.legend(); plt.show()

def caption(n):
    t = "startseq"
    for _ in range(L):
        p = model.predict([feats[n][None], pad_sequences(tok.texts_to_sequences([t]), maxlen=L)], verbose=0).argmax()
        w = tok.index_word.get(p)
        if w in (None, "endseq"): break
        t += " " + w
    return t.replace("startseq", "").strip()

n = test_n[0]
plt.imshow(load_img(f"{img_dir}/{n}")); plt.title(caption(n)); plt.axis("off"); plt.show()